# 02. Structure Prediction (ColabFold)

## Objective
To obtain and analyze the 3D structural model of **HPAG1_0576**. 

## Methodology
While the AlphaFold Protein Structure Database (AFDB) provides pre-computed models, we have performed a custom **ColabFold** (AlphaFold2 with MMseqs2) run to obtain high-resolution predictions with custom parameters (6 recycles, pTM head enabled). This allows for a more detailed assessment of the protein's fold and confidence metrics.

In [ ]:
import os
import json
from IPython.display import Image, display

# 1. Define paths to organized ColabFold results
uniprot_id = "HPAG1_0576"
model_path = "../results/models/HPAG1_0576_colabfold_relaxed.pdb"
figures_dir = "../results/figures/colabfold"
tables_dir = "../results/tables/colabfold"

# Verify file existence
if os.path.exists(model_path):
    print(f"[SUCCESS] Found Rank 1 structural model: {model_path}")
else:
    print(f"[ERROR] Model file not found at {model_path}. Please check the organization step.")

## Structural Confidence Metrics
ColabFold generates several diagnostic plots to assess the quality of the prediction:
1. **MSA Coverage:** Shows the depth of the multiple sequence alignment used for prediction.
2. **pLDDT:** Predicted Local Distance Difference Test (per-residue confidence).
3. **PAE:** Predicted Aligned Error (confidence in relative domain orientations).

In [ ]:
# 2. Display ColabFold Visuals
plots = [
    ("MSA Coverage", "HPAG1_0576_f1e48_coverage.png"),
    ("pLDDT Scores", "HPAG1_0576_f1e48_plddt.png"),
    ("Predicted Aligned Error (PAE)", "HPAG1_0576_f1e48_pae.png")
]

for title, filename in plots:
    path = os.path.join(figures_dir, filename)
    if os.path.exists(path):
        print(f"### {title}")
        display(Image(filename=path))
    else:
        print(f"[WARNING] Plot {filename} not found in {figures_dir}")

## 3D Visualization
We visualize the predicted structure colored by **pLDDT** confidence scores:
- **Blue (>90):** Very High confidence.
- **Light Blue (70-90):** Confident (Backbone likely correct).
- **Yellow (50-70):** Low confidence.
- **Orange (<50):** Very Low confidence (Often represents disordered regions).

In [ ]:
import py3Dmol

def visualize_protein(pdb_file):
    with open(pdb_file, 'r') as f:
        pdb_data = f.read()
    
    view = py3Dmol.view(width=800, height=600)
    view.addModel(pdb_data, 'pdb')
    
    # Color by B-factor (pLDDT)
    view.setStyle({'cartoon': {'colorscheme': {'prop': 'b', 'gradient': 'roygb', 'min': 50, 'max': 90}}})
    
    view.zoomTo()
    return view

view = visualize_protein(model_path)
view.show()

In [ ]:
# 4. Extract and Save Summary Metrics
plddt_values = []
with open(model_path, 'r') as f:
    for line in f:
        if line.startswith("ATOM"):
            plddt = float(line[60:66].strip())
            plddt_values.append(plddt)

average_plddt = sum(plddt_values) / len(plddt_values)

# Note: pTM score is retrieved from the ColabFold log/JSON
ptm_score = 0.753 # Value from Rank 1 model in log.txt

metrics = {
    "protein_id": "HPAG1_0576",
    "average_plddt": round(average_plddt, 2),
    "ptm_score": ptm_score,
    "model_source": "ColabFold (AlphaFold2-ptm)",
    "status": "Confident Fold" if average_plddt > 70 else "Low Confidence"
}

with open(f"../results/tables/colabfold_summary_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print(f"Average pLDDT: {average_plddt:.2f}")
print(f"pTM Score: {ptm_score}")

## Results Summary
The structure prediction for **HPAG1_0576** shows a high-confidence global fold (**pLDDT: 83.1**, **pTM: 0.753**). 

**Key Output Files:**
- `results/models/HPAG1_0576_colabfold_relaxed.pdb`: Primary 3D model.
- `results/figures/colabfold/`: Diagnostic plots.
- `results/tables/colabfold_summary_metrics.json`: Final confidence scores.

Next, we will proceed to **Notebook 03** for detailed physicochemical profiling and functional site analysis.